In [ ]:
import os
import cv2
import numpy as np
from deepface import DeepFace

# The main data directory, expected to be in the same location as this script.
DATA_DIRECTORY = 'data'
SUPPORTED_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff')


def load_dataset(base_path, splits):
    """
    Loads the dataset from a directory structure like:
    base_path/
    ├── train/
    │   ├── happy/
    │   │   ├── img1.jpg
    │   │   └── ...
    │   └── sad/
    │       └── ...
    └── test/
        └── ...
    
    Args:
        base_path (str): The path to the main data folder (e.g., 'data').
        splits (list): A list of the splits to load (e.g., ['train', 'test']).

    Returns:
        dict: A dictionary where keys are the splits and values are lists of tuples.
              Each tuple contains (image_path, image_data, label).
    """
    all_data = {}
    print(f"--- Loading Dataset from '{base_path}' ---")

    for split in splits:
        split_path = os.path.join(base_path, split)
        if not os.path.isdir(split_path):
            print(f"Warning: Split directory not found, skipping: '{split_path}'")
            continue

        print(f"\n- Loading '{split}' split...")
        images_with_labels = []
        
        # Discover emotion subdirectories automatically
        emotion_folders = [d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d))]
        if not emotion_folders:
            print(f"  No emotion subdirectories found in '{split_path}'.")
            continue
            
        for emotion_label in emotion_folders:
            emotion_path = os.path.join(split_path, emotion_label)
            print(f"  Loading images for label: '{emotion_label}'")
            
            for filename in os.listdir(emotion_path):
                if filename.lower().endswith(SUPPORTED_EXTENSIONS):
                    file_path = os.path.join(emotion_path, filename)
                    img = cv2.imread(file_path)
                    if img is not None:
                        images_with_labels.append((file_path, img, emotion_label))
                    else:
                        print(f"    Warning: Could not load image '{filename}'.")
        
        all_data[split] = images_with_labels
        
    return all_data


def preprocess_image_for_model(image_data):
    """
    Placeholder function for image preprocessing.
    In a real scenario, you would resize, normalize, and format the image.
    """
    # This function remains the same.
    # A real implementation would resize to model's expected input size, e.g., (48, 48)
    # and normalize pixel values.
    return image_data


def predict_emotion(preprocessed_image):
    try:
        # The 'actions' parameter specifies which analysis to perform.
        # 'enforce_detection=False' prevents DeepFace from throwing an error if no face is found.
        analysis_result = DeepFace.analyze(
            img_path=image_data,
            actions=['emotion'],
            enforce_detection=False
        )

        # DeepFace.analyze returns a list of dictionaries, one for each detected face.
        # We will use the result for the first face.
        if isinstance(analysis_result, list) and len(analysis_result) > 0:
            # Check if a face was actually detected and analyzed
            if analysis_result[0]['dominant_emotion'] is not None:
                return analysis_result[0]['dominant_emotion']
            else:
                return "no_face_detected"
        else:
             # This case handles unexpected output formats from DeepFace
            return "analysis_failed"

    except Exception as e:
        # Catch any other unexpected errors during analysis
        print(f"An error occurred during emotion analysis: {e}")
        return "error"


def main():
    """Main function to load data, process images, and evaluate predictions."""
    data_subdirectories = ['train', 'test']

    # Load data from the 'data' directory which contains 'train' and 'test' folders,
    # which in turn contain emotion-labeled subfolders.
    all_loaded_data = load_dataset(DATA_DIRECTORY, data_subdirectories)

    if not all_loaded_data or not any(all_loaded_data.values()):
        print("\nNo data loaded. Please ensure a 'data' folder exists with 'train' and 'test' subfolders.")
        print("Each of these should contain folders named after emotions (e.g., 'happy', 'sad').")
        print("Example structure: your_project/data/train/happy/image1.jpg")
        return

    print("\n--- Processing and Evaluating Images ---")
    for data_split, image_files in all_loaded_data.items():
        if image_files:
            print(f"\n--- Evaluating {data_split.upper()} set ({len(image_files)} images) ---")
            correct_predictions = 0
            
            for img_path, img_data, true_label in image_files:
                try:
                    # 1. Preprocess the image
                    preprocessed_img = preprocess_image_for_model(img_data)

                    # 2. Get a prediction from the model
                    predicted_label = predict_emotion(preprocessed_img)

                    # 3. Compare prediction with the true label
                    print(f"  Processing '{os.path.basename(img_path)}' | True Label: {true_label:<10} | Predicted: {predicted_label}")
                    if predicted_label.lower() == true_label.lower():
                        correct_predictions += 1
                        
                except Exception as e:
                    print(f"    Error processing image '{os.path.basename(img_path)}': {e}")
            
            # Calculate and print accuracy for the current data split
            total_images = len(image_files)
            accuracy = (correct_predictions / total_images) * 100 if total_images > 0 else 0
            print(f"\n  Accuracy for {data_split.upper()} set: {accuracy:.2f}% ({correct_predictions}/{total_images} correct)")
            
        else:
            print(f"\n--- No images found in {data_split.upper()} set ---")

    print("\n--- Processing Complete ---")


if __name__ == "__main__":
    main()

--- Loading Dataset from 'data' ---

- Loading 'train' split...
  Loading images for label: 'angry'
  Loading images for label: 'disgust'
  Loading images for label: 'fear'
  Loading images for label: 'happy'
  Loading images for label: 'neutral'
  Loading images for label: 'sad'
  Loading images for label: 'surprise'

- Loading 'test' split...
  Loading images for label: 'angry'
  Loading images for label: 'disgust'
  Loading images for label: 'fear'
  Loading images for label: 'happy'
  Loading images for label: 'neutral'
  Loading images for label: 'sad'
  Loading images for label: 'surprise'

--- Processing and Evaluating Images ---

--- Evaluating TRAIN set (34068 images) ---
  Processing '0.jpg' | True Label: angry      | Predicted: disgust
  Processing '1.jpg' | True Label: angry      | Predicted: sad
  Processing '10.jpg' | True Label: angry      | Predicted: surprise
  Processing '10002.jpg' | True Label: angry      | Predicted: disgust
  Processing '10016.jpg' | True Label: a